# Calibrazione del motore di decisione (Utente 1)

Obiettivo: stimare i parametri di `sentinel/model.py` (pesi e soglia) da fonti **pertinenti alle truffe agli anziani**, e mostrare la curva ROC e l'analisi di sensibilità.

**Fonti da usare**
- Statistica criminale di polizia (UST): reati di truffa, andamento annuale
- Rapporti NCSC sulle chiamate di falsi poliziotti e falsi nipoti
- Cronaca locale (tio.ch, RSI): importi tipici e copione delle truffe

**Da non usare:** fonti sulle violazioni di dati aziendali (VERIS, IBM Cost of a Data Breach).

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd
from sentinel.profiles import PROFILI, genera_storico, inietta_frode
from sentinel.model import adatta_baseline, punteggia, PESI
from sentinel.economics import curva_costo, soglia_ottimale

## 1. Parametri dalle fonti
Riportare qui, con la fonte accanto: importo tipico per tipo di truffa, frequenza annua, quota di casi con prelievo di contanti.

In [ ]:
parametri = {
    # 'importo_medio_truffa_contanti': (valore, 'fonte'),
}

## 2. Portafoglio simulato di molti clienti
Un cliente non basta: simulare ad esempio 500 clienti per profilo, con frodi rare, e calcolare la curva ROC.

In [ ]:
def portafoglio(profilo, n=200, prob_frode=0.02, seed=0):
    rng = np.random.default_rng(seed)
    pezzi = []
    for i in range(n):
        st = genera_storico(profilo, giorni=365, seed=seed + i)
        base = adatta_baseline(st)
        df = inietta_frode(st, profilo, n=1, seed=seed + i) if rng.random() < prob_frode else st
        pezzi.append(punteggia(df, base).assign(cliente=i))
    return pd.concat(pezzi, ignore_index=True)

## 3. Curva ROC e analisi di sensibilità
Come si sposta la soglia ottimale al variare del costo di un falso allarme (disturbo al contatto) e dell'importo delle frodi?

In [ ]:
# df = portafoglio('truffa_esterna')
# for costo in [1, 5, 20]:
#     print(costo, soglia_ottimale(curva_costo(df, costo_falso_allarme=costo)))

## 4. (Avanzato) Controprova con Isolation Forest